[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week7/diffusion_demo.ipynb)

# Diffusion models -- from noise to images

**PSYC 51.17: Models of language and communication**
**Week 7**

---

## Overview

In this notebook we will:
1. Implement the forward diffusion process from scratch
2. Train a tiny denoiser on synthetic 2D data
3. Generate images using HuggingFace Diffusers
4. Explore classifier-free guidance
5. Reflect on ethical implications of generative models

In [ ]:
# Install required packages (for Colab)
!pip install -q matplotlib numpy torch torchvision diffusers transformers accelerate scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---

## Part 1: The forward diffusion process

The key insight of diffusion models is simple: if we know how to
*destroy* data by adding noise, we can learn to *reverse* that process.

The forward process gradually adds Gaussian noise to data over $ timesteps:

6715q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t; \sqrt{1 - eta_t}\,\mathbf{x}_{t-1},\; eta_t\mathbf{I})6715

Thanks to the reparameterization trick, we can jump directly to any timestep:

6715\mathbf{x}_t = \sqrt{ar{lpha}_t}\,\mathbf{x}_0 + \sqrt{1 - ar{lpha}_t}\,oldsymbol{\epsilon}, \quad oldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})6715

In [ ]:
def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    """Linear noise schedule (Ho et al., 2020)."""
    betas = np.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bar = np.cumprod(alphas)
    return betas, alpha_bar

def cosine_schedule(T, s=0.008):
    """Cosine noise schedule (Nichol & Dhariwal, 2021)."""
    steps = np.arange(T + 1)
    f = np.cos(((steps / T) + s) / (1 + s) * np.pi / 2) ** 2
    alpha_bar = f / f[0]
    alpha_bar = np.clip(alpha_bar[1:], 1e-6, 1.0)
    betas = 1 - alpha_bar / np.concatenate([[1.0], alpha_bar[:-1]])
    betas = np.clip(betas, 0, 0.999)
    return betas, alpha_bar

T = 1000
_, ab_linear = linear_schedule(T)
_, ab_cosine = cosine_schedule(T)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ab_linear, color="#003C6C", linewidth=2, label="Linear")
ax.plot(ab_cosine, color="#267300", linewidth=2, label="Cosine")
ax.set_xlabel("Timestep t", fontsize=12)
ax.set_ylabel(r"$ar{lpha}_t$ (signal remaining)", fontsize=12)
ax.set_title("Noise schedules", fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, T)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Visualizing the forward process on a 2D point cloud

In [ ]:
from sklearn.datasets import make_moons

# Generate 2D data (two moons)
X, _ = make_moons(n_samples=2000, noise=0.05)
X = (X - X.mean(axis=0)) / X.std(axis=0)  # normalize

_, alpha_bar = cosine_schedule(1000)

timesteps = [0, 100, 300, 500, 700, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(18, 3))

for i, t in enumerate(timesteps):
    eps = np.random.randn(*X.shape)
    if t == 0:
        x_t = X
    else:
        x_t = np.sqrt(alpha_bar[t]) * X + np.sqrt(1 - alpha_bar[t]) * eps

    axes[i].scatter(x_t[:, 0], x_t[:, 1], s=2, alpha=0.5, color="#003C6C")
    axes[i].set_title(f"t = {t}", fontsize=12)
    axes[i].set_xlim(-3.5, 3.5)
    axes[i].set_ylim(-3.5, 3.5)
    axes[i].set_aspect("equal")
    axes[i].set_xticks([])
    axes[i].set_yticks([])

fig.suptitle("Forward diffusion on 2D moons", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Part 2: Learning to reverse the process

Now we train a small neural network to predict the noise that was added.
Given noisy data $\mathbf{x}_t$ and timestep $, the network learns:

6715oldsymbol{\epsilon}_	heta(\mathbf{x}_t, t) pprox oldsymbol{\epsilon}6715

We use the simplified DDPM loss:

6715\mathcal{L} = \mathbb{E}_{t, \mathbf{x}_0, oldsymbol{\epsilon}} \left[ \| oldsymbol{\epsilon} - oldsymbol{\epsilon}_	heta(\mathbf{x}_t, t) \|^2 ight]6715

In [ ]:
class SimpleDenoiser(nn.Module):
    """Small MLP that predicts noise given (x_t, t)."""

    def __init__(self, data_dim=2, hidden=128, T=1000):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x_t, t):
        t_emb = self.time_embed(t)
        inp = torch.cat([x_t, t_emb], dim=-1)
        return self.net(inp)

model = SimpleDenoiser(data_dim=2, hidden=128, T=1000)
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Prepare data and schedules as tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
betas_t, ab_t = cosine_schedule(1000)
alpha_bar_t = torch.tensor(ab_t, dtype=torch.float32)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
losses = []

for step in range(3000):
    # Sample random batch
    idx = torch.randint(0, len(X_tensor), (256,))
    x0 = X_tensor[idx]

    # Sample random timesteps
    t = torch.randint(0, 1000, (256,))

    # Sample noise
    eps = torch.randn_like(x0)

    # Create noisy data
    ab = alpha_bar_t[t].unsqueeze(1)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps

    # Predict noise
    eps_pred = model(x_t, t)
    loss = nn.functional.mse_loss(eps_pred, eps)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % 1000 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

plt.figure(figsize=(6, 3))
plt.plot(losses, color="#003C6C", alpha=0.3, linewidth=0.5)
# Smoothed
window = 50
smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
plt.plot(range(window-1, len(losses)), smoothed, color="#003C6C", linewidth=2)
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title("Training the denoiser")
plt.tight_layout()
plt.show()

### Sampling: running the reverse process

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n_samples=2000, T=1000):
    """DDPM sampling loop."""
    betas_np, ab_np = cosine_schedule(T)
    betas = torch.tensor(betas_np, dtype=torch.float32)
    alphas = 1 - betas
    alpha_bar = torch.tensor(ab_np, dtype=torch.float32)

    # Start from pure noise
    x = torch.randn(n_samples, 2)
    trajectory = [x.numpy().copy()]

    for t_val in reversed(range(T)):
        t = torch.full((n_samples,), t_val, dtype=torch.long)
        eps_pred = model(x, t)

        # DDPM update
        alpha_t = alphas[t_val]
        ab_t = alpha_bar[t_val]
        coef = betas[t_val] / torch.sqrt(1 - ab_t)
        mean = (1 / torch.sqrt(alpha_t)) * (x - coef * eps_pred)

        if t_val > 0:
            sigma = torch.sqrt(betas[t_val])
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean

        if t_val in [999, 750, 500, 250, 100, 0]:
            trajectory.append(x.numpy().copy())

    return x.numpy(), trajectory

samples, trajectory = sample_ddpm(model, n_samples=2000)
print("✓ Sampling complete!")

fig, axes = plt.subplots(1, len(trajectory), figsize=(18, 3))
labels = ["t=1000", "t=999", "t=750", "t=500", "t=250", "t=100", "t=0"]
for i, (pts, lbl) in enumerate(zip(trajectory, labels)):
    axes[i].scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5, color="#003C6C")
    axes[i].set_title(lbl, fontsize=12)
    axes[i].set_xlim(-3.5, 3.5)
    axes[i].set_ylim(-3.5, 3.5)
    axes[i].set_aspect("equal")
    axes[i].set_xticks([])
    axes[i].set_yticks([])

fig.suptitle("Reverse diffusion: noise to moons", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Part 3: Generating images with HuggingFace Diffusers

Now let us use a pretrained diffusion model to generate real images.
We will use the DDPM model trained on CIFAR-10 (32x32 images).

In [ ]:
from diffusers import DDPMPipeline

# Load a small pretrained model (no auth required)
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32")
print("✓ Model loaded!")

# Generate a batch of images
images = pipe(batch_size=16, num_inference_steps=100).images

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.axis("off")
fig.suptitle("Generated CIFAR-10 images (DDPM)", fontsize=14)
plt.tight_layout()
plt.show()

### Manual sampling loop

Let us peek inside the pipeline and run the denoising loop manually:

In [ ]:
from diffusers import DDPMScheduler, UNet2DModel

scheduler = DDPMScheduler.from_pretrained("google/ddpm-cifar10-32")
unet = UNet2DModel.from_pretrained("google/ddpm-cifar10-32")

# Start from random noise
sample = torch.randn(1, 3, 32, 32)
scheduler.set_timesteps(100)

snapshots = {}
for i, t in enumerate(scheduler.timesteps):
    with torch.no_grad():
        noise_pred = unet(sample, t).sample
    sample = scheduler.step(noise_pred, t, sample).prev_sample

    if i in [0, 25, 50, 75, 99]:
        img = (sample.squeeze().permute(1, 2, 0).numpy() + 1) / 2
        snapshots[f"Step {i}"] = np.clip(img, 0, 1)

fig, axes = plt.subplots(1, len(snapshots), figsize=(12, 3))
for ax, (label, img) in zip(axes, snapshots.items()):
    ax.imshow(img)
    ax.set_title(label, fontsize=11)
    ax.axis("off")
fig.suptitle("Manual denoising loop", fontsize=14)
plt.tight_layout()
plt.show()

---

## Part 4: Text-to-image with classifier-free guidance

If a GPU is available, we can try Stable Diffusion.
Classifier-free guidance controls how strongly the model follows the text prompt:

6715\hat{oldsymbol{\epsilon}} = oldsymbol{\epsilon}_	heta(\mathbf{x}_t, arnothing) + w \cdot \left(oldsymbol{\epsilon}_	heta(\mathbf{x}_t, c) - oldsymbol{\epsilon}_	heta(\mathbf{x}_t, arnothing)ight)6715

where $ is the guidance scale. Higher values produce images that match
the prompt more closely but with less diversity.

In [ ]:
if torch.cuda.is_available():
    from diffusers import StableDiffusionPipeline

    pipe_sd = StableDiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-2-1-base",
        torch_dtype=torch.float16,
    ).to("cuda")

    prompt = "A watercolor painting of a mountain lake at sunset"
    guidance_scales = [1.0, 3.0, 7.5, 15.0]

    fig, axes = plt.subplots(1, len(guidance_scales), figsize=(16, 4))
    for ax, gs in zip(axes, guidance_scales):
        img = pipe_sd(prompt, guidance_scale=gs, num_inference_steps=30).images[0]
        ax.imshow(img)
        ax.set_title(f"w = {gs}", fontsize=12)
        ax.axis("off")
    fig.suptitle(f"Guidance scale comparison
"{prompt}"", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No GPU detected. To try text-to-image generation:")
    print("  1. Go to Runtime > Change runtime type > GPU")
    print("  2. Re-run this cell")
    print()
    print("Effect of guidance scale (w):")
    print("  w = 1.0  : Ignores prompt, maximum diversity")
    print("  w = 3.0  : Mild prompt following")
    print("  w = 7.5  : Default, good balance (most common)")
    print("  w = 15.0 : Strong prompt adherence, less diversity")

---

## Part 5: Discussion questions

1. **Forward vs. reverse**: Why is the forward process easy but the reverse process hard? What makes learning the reverse process tractable?

2. **Noise prediction**: Why does predicting the *noise* work better than directly predicting the clean image? (Hint: think about what the network needs to represent at each timestep.)

3. **Schedule design**: Looking at the noise schedule plots from Part 1, why does the cosine schedule tend to produce better results than the linear schedule?

4. **Comparison to autoregressive models**: How does the generation process in diffusion models differ from autoregressive models like GPT? What are the tradeoffs in terms of speed, quality, and controllability?

5. **Ethical considerations**: Diffusion models can generate photorealistic images from text. Consider:
   - What are the potential harms of this technology?
   - Who should be responsible when AI-generated images cause harm?
   - Should there be watermarking requirements for AI-generated content?
   - How do you balance open research with preventing misuse?

6. **Creative applications**: Beyond image generation, what other domains could benefit from the diffusion framework? Think about audio, video, molecular design, or other structured data.

---

*Notebook for PSYC 51.17, Dartmouth College. Based on concepts from [Ho et al. (2020)](https://arxiv.org/abs/2006.11239) and [Rombach et al. (2022)](https://arxiv.org/abs/2112.10752).*